In [ ]:
!pip install pillow

zsh:1: command not found: pip


In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
from PIL import Image, ImageTk
import json
from urllib.request import urlopen
from urllib.error import URLError
import datetime
import io
from urllib.request import urlopen as uReq

class WeatherApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Advanced Weather App")
        self.root.geometry("600x700")
        self.root.resizable(False, False)
        
        # Dark theme colors
        self.bg_color = '#2d3436'
        self.fg_color = '#dfe6e9'
        self.card_bg = '#3d4548'
        self.text_color = '#b2bec3'
        
        # Weather condition colors
        self.weather_colors = {
            'clear': '#fdcb6e',       # Sunny
            'clouds': '#b2bec3',      # Cloudy
            'rain': '#74b9ff',        # Rainy
            'thunderstorm': '#a55eea', # Thunder
            'snow': '#dfe6e9',        # Snow
            'mist': '#636e72',        # Fog/Mist
            'default': '#0984e3'      # Default
        }
        
        self.root.configure(bg=self.bg_color)
        
        # API configuration
        self.api_key = "56fdf7c8c02913f9dabf9e1ea5935e02"
        self.base_url = "http://api.openweathermap.org/data/2.5/weather?"
        self.icon_url = "http://openweathermap.org/img/wn/{}@2x.png"
        
        # Create main container
        self.main_frame = tk.Frame(root, bg=self.bg_color, padx=20, pady=20)
        self.main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Header
        self.header = tk.Label(self.main_frame, 
                             text="Weather Forecast", 
                             font=('Helvetica', 24, 'bold'), 
                             bg=self.bg_color,
                             fg=self.fg_color)
        self.header.pack(pady=(0, 20))
        
        # Search frame
        self.search_frame = tk.Frame(self.main_frame, bg=self.bg_color)
        self.search_frame.pack(fill=tk.X, pady=(0, 20))
        
        tk.Label(self.search_frame, 
                text="Enter City or ZIP Code:", 
                font=('Helvetica', 12), 
                bg=self.bg_color,
                fg=self.fg_color).pack(side=tk.LEFT, padx=(0, 10))
        
        self.city_entry = ttk.Entry(self.search_frame, font=('Helvetica', 12), width=25)
        self.city_entry.pack(side=tk.LEFT, padx=(0, 10))
        
        self.search_btn = ttk.Button(self.search_frame, 
                                   text="Search", 
                                   command=self.get_weather)
        self.search_btn.pack(side=tk.LEFT)
        
        # Weather display frame
        self.weather_frame = tk.Frame(self.main_frame, 
                                     bg=self.card_bg, 
                                     bd=2, 
                                     relief=tk.GROOVE,
                                     padx=20,
                                     pady=20)
        self.weather_frame.pack(fill=tk.BOTH, expand=True)
        
        # Initialize weather display widgets
        self.init_weather_widgets()
        
        # Configure styles
        self.configure_styles()
        
        # Bind Enter key to search
        self.root.bind('<Return>', lambda event: self.get_weather())
    
    def configure_styles(self):
        style = ttk.Style()
        style.theme_use('clam')
        
        # Configure main colors
        style.configure('.', 
                       background=self.bg_color,
                       foreground=self.fg_color,
                       fieldbackground=self.card_bg)
        
        # Configure buttons
        style.configure('TButton', 
                       font=('Helvetica', 12),
                       foreground=self.fg_color,
                       background=self.weather_colors['default'],
                       bordercolor=self.weather_colors['default'])
        style.map('TButton',
                 background=[('active', '#0767b1')])
        
        # Configure entry
        style.configure('TEntry', 
                       font=('Helvetica', 12),
                       padding=5,
                       fieldbackground=self.card_bg,
                       foreground=self.fg_color)
    
    def init_weather_widgets(self):
        # Weather icon
        self.icon_label = tk.Label(self.weather_frame, bg=self.card_bg)
        self.icon_label.pack(pady=(10, 5))
        
        # Location and date
        self.location_label = tk.Label(self.weather_frame, 
                                      font=('Helvetica', 18, 'bold'), 
                                      bg=self.card_bg,
                                      fg=self.fg_color)
        self.location_label.pack(pady=(0, 10))
        
        self.date_label = tk.Label(self.weather_frame, 
                                  font=('Helvetica', 12), 
                                  bg=self.card_bg,
                                  fg=self.text_color)
        self.date_label.pack(pady=(0, 20))
        
        # Temperature display
        self.temp_label = tk.Label(self.weather_frame, 
                                  font=('Helvetica', 48, 'bold'), 
                                  bg=self.card_bg,
                                  fg=self.fg_color)
        self.temp_label.pack(pady=(0, 10))
        
        # Weather description
        self.desc_label = tk.Label(self.weather_frame, 
                                  font=('Helvetica', 16), 
                                  bg=self.card_bg,
                                  fg=self.text_color)
        self.desc_label.pack(pady=(0, 20))
        
        # Additional details frame
        self.details_frame = tk.Frame(self.weather_frame, bg=self.card_bg)
        self.details_frame.pack(fill=tk.X, pady=(0, 20))
        
        # Left details column
        self.left_details = tk.Frame(self.details_frame, bg=self.card_bg)
        self.left_details.pack(side=tk.LEFT, expand=True)
        
        # Right details column
        self.right_details = tk.Frame(self.details_frame, bg=self.card_bg)
        self.right_details.pack(side=tk.RIGHT, expand=True)
        
        # Initialize detail labels
        self.feels_like_label = tk.Label(self.left_details, 
                                         font=('Helvetica', 12), 
                                         bg=self.card_bg,
                                         fg=self.text_color,
                                         anchor='w')
        self.feels_like_label.pack(fill=tk.X, pady=2)
        
        self.humidity_label = tk.Label(self.left_details, 
                                      font=('Helvetica', 12), 
                                      bg=self.card_bg,
                                      fg=self.text_color,
                                      anchor='w')
        self.humidity_label.pack(fill=tk.X, pady=2)
        
        self.pressure_label = tk.Label(self.left_details, 
                                      font=('Helvetica', 12), 
                                      bg=self.card_bg,
                                      fg=self.text_color,
                                      anchor='w')
        self.pressure_label.pack(fill=tk.X, pady=2)
        
        self.wind_label = tk.Label(self.right_details, 
                                  font=('Helvetica', 12), 
                                  bg=self.card_bg,
                                  fg=self.text_color,
                                  anchor='w')
        self.wind_label.pack(fill=tk.X, pady=2)
        
        self.visibility_label = tk.Label(self.right_details, 
                                        font=('Helvetica', 12), 
                                        bg=self.card_bg,
                                        fg=self.text_color,
                                        anchor='w')
        self.visibility_label.pack(fill=tk.X, pady=2)
        
        self.sunrise_label = tk.Label(self.right_details, 
                                     font=('Helvetica', 12), 
                                     bg=self.card_bg,
                                     fg=self.text_color,
                                     anchor='w')
        self.sunrise_label.pack(fill=tk.X, pady=2)
        
        # Initially hide weather frame
        self.weather_frame.pack_forget()
    
    def get_weather(self):
        city = self.city_entry.get().strip()
        if not city:
            messagebox.showerror("Error", "Please enter a city name or ZIP code")
            return
        
        try:
            url = f"{self.base_url}q={city}&appid={self.api_key}&units=metric"
            with urlopen(url) as response:
                data = json.loads(response.read().decode())
            
            if data["cod"] != 200:
                messagebox.showerror("Error", data["message"])
                return
            
            self.display_weather(data)
            
        except URLError:
            messagebox.showerror("Error", "Failed to connect to weather service. Please check your internet connection.")
        except Exception as e:
            messagebox.showerror("Error", f"An error occurred: {str(e)}")
    
    def display_weather(self, data):
        # Show weather frame if hidden
        if not self.weather_frame.winfo_ismapped():
            self.weather_frame.pack(fill=tk.BOTH, expand=True)
        
        # Get weather condition
        weather_condition = data["weather"][0]["main"].lower()
        accent_color = self.weather_colors.get(weather_condition, self.weather_colors['default'])
        
        # Update accent colors
        self.update_accent_colors(accent_color)
        
        # Get weather icon
        icon_code = data["weather"][0]["icon"]
        self.load_weather_icon(icon_code, accent_color)
        
        # Location and date
        location = f"{data['name']}, {data['sys']['country']}"
        self.location_label.config(text=location)
        
        current_date = datetime.datetime.now().strftime("%A, %B %d, %Y %I:%M %p")
        self.date_label.config(text=current_date)
        
        # Temperature and description
        temp = data["main"]["temp"]
        self.temp_label.config(text=f"{temp}°C", fg=accent_color)
        
        description = data["weather"][0]["description"].capitalize()
        self.desc_label.config(text=description)
        
        # Additional details
        feels_like = data["main"]["feels_like"]
        humidity = data["main"]["humidity"]
        pressure = data["main"]["pressure"]
        wind_speed = data["wind"]["speed"]
        visibility = data.get("visibility", "N/A")
        sunrise = datetime.datetime.fromtimestamp(data["sys"]["sunrise"]).strftime("%I:%M %p")
        sunset = datetime.datetime.fromtimestamp(data["sys"]["sunset"]).strftime("%I:%M %p")
        
        self.feels_like_label.config(text=f"Feels like: {feels_like}°C")
        self.humidity_label.config(text=f"Humidity: {humidity}%")
        self.pressure_label.config(text=f"Pressure: {pressure} hPa")
        self.wind_label.config(text=f"Wind: {wind_speed} m/s")
        
        if visibility != "N/A":
            visibility = f"{visibility/1000} km" if visibility >= 1000 else f"{visibility} m"
        self.visibility_label.config(text=f"Visibility: {visibility}")
        
        self.sunrise_label.config(text=f"Sunrise: {sunrise} | Sunset: {sunset}")
    
    def update_accent_colors(self, color):
        # Update button colors
        style = ttk.Style()
        style.configure('TButton', 
                       background=color,
                       bordercolor=color)
        style.map('TButton',
                 background=[('active', self.darken_color(color))])
        
        # Update other accent elements as needed
    
    def darken_color(self, color, factor=0.8):
        """Darken a hex color by a given factor"""
        color = color.lstrip('#')
        rgb = tuple(int(color[i:i+2], 16) for i in (0, 2, 4))
        darkened = tuple(max(0, int(c * factor)) for c in rgb)
        return '#%02x%02x%02x' % darkened
    
    def load_weather_icon(self, icon_code, bg_color=None):
        try:
            # Get image from URL
            with uReq(self.icon_url.format(icon_code)) as u:
                image_data = u.read()
            
            # Convert to PhotoImage
            image = Image.open(io.BytesIO(image_data))
            image = image.resize((100, 100), Image.LANCZOS)
            
            # Add colored background to the icon
            if bg_color:
                background = Image.new('RGBA', image.size, bg_color)
                background.paste(image, (0, 0), image)
                image = background
            
            photo = ImageTk.PhotoImage(image)
            
            # Update label
            self.icon_label.config(image=photo)
            self.icon_label.image = photo  # Keep reference
            
        except Exception as e:
            print(f"Error loading icon: {e}")

if __name__ == "__main__":
    root = tk.Tk()
    app = WeatherApp(root)
    root.mainloop()